In [39]:
%pip install -q python-dotenv openai openpyxl langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


In [40]:
# imports
import os
import time

import pandas as pd
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI


In [41]:
# constants
load_dotenv()

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"
PDFS_PATH = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"
RAG_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
JUDGE_MODEL = ""
VECTORSTORE_DIR = "financebench_rag_faiss"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


In [42]:
# dataset
df = (
    pd.read_json(
        "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl", lines=True
    )
    .sort_values(by="financebench_id", ascending=True)
    .reset_index(drop=True)
)

# For each row, replace the "doc_link" value with the url from "https://github.com/patronus-ai/financebench/tree/main/pdfs" such that the url is: "https://github.com/patronus-ai/financebench/tree/main/pdfs/{{doc_name}}.pdf"
df["doc_link"] = df["doc_name"].apply(lambda x: f"{PDFS_PATH}/{x}.pdf")

print("columns:", df.columns.tolist())
df.head(2)

columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://raw.githubusercontent.com/patronus-ai/...


---
## Task 1 - Naive Generation

In [43]:
# answer the first 5 questions of each question_type - 5 domain-relevant, 5 novel-generated

assignment2_naive_generation_filename = "assignment2_naive_generation.xlsx"


if os.path.exists(assignment2_naive_generation_filename):
    print("Loading existing results...")
    answers_df = pd.read_excel(assignment2_naive_generation_filename)
else:
    # Select the first 5 questions for each question_type
    questions_domain = df[df["question_type"] == "domain-relevant"].head(5)
    questions_novel = df[df["question_type"] == "novel-generated"].head(5)
    selected_questions = pd.concat([questions_domain, questions_novel]).reset_index(
        drop=True
    )

    answers = []

    for idx, row in selected_questions.iterrows():
        prompt = f"""
Answer the question in 2-4 sentences.
If you don't know the answer, say "I don't know".
QUESTION: {row["question"]}
ANSWER:
"""
        response = client.chat.completions.create(
            model=RAG_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=200,
        )
        naive_answer = response.choices[0].message.content.strip()
        result = pd.Series(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": naive_answer,
                "ground_truth": row["answer"],
                "verdict": "",  # correct/partially correct/wrong/refused
            }
        )
        answers.append(result)
        time.sleep(1.5)  # rate limiting

    answers_df = pd.DataFrame(answers)

    # save results before setting the verdict
    answers_df.to_excel(assignment2_naive_generation_filename, index=False)

Loading existing results...


In [110]:
# set verdict
VERDICT_MAP = {
    1: "correct",
    2: "partially correct",
    3: "wrong",
    4: "refused",
}
verdicts_dict = {
    "financebench_id_00005": VERDICT_MAP[1],
    "financebench_id_00070": VERDICT_MAP[4],
    "financebench_id_00080": VERDICT_MAP[1],
    "financebench_id_00206": VERDICT_MAP[1],
    "financebench_id_00215": VERDICT_MAP[2],
    "financebench_id_00283": VERDICT_MAP[3],
    "financebench_id_00288": VERDICT_MAP[4],
    "financebench_id_00299": VERDICT_MAP[4],
    "financebench_id_00302": VERDICT_MAP[4],
    "financebench_id_00382": VERDICT_MAP[2],
}

for fid, verdict in verdicts_dict.items():
    answers_df.loc[answers_df["financebench_id"] == fid, "verdict"] = verdict

answers_df.to_excel(assignment2_naive_generation_filename, index=False)

In [44]:
answers_df.head(2)

,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",Yes. Corning had a positive working capital am...,correct
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,I don't know the specific details of American ...,"No, American Water Works had negative working ...",refused


#### Questions:

1. Cases where the model refused or asked for more information - why?
- In `novel-generated` Q.s, the model refused to answer (not enough knowledge) 3 times as opposed to only once for the given `domain-relevant` Q.s.<br>
I don't see a reason why sometimes it refuses to answer while sometime it hallucinates some answer...<br>
However, it DOES refuse because we told it in the prompt ('If you don't know the answer, say "I don't know"') - and it really does not have any relevant information for any of these questions.

2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
- Overall, the model is confident in it's answers - regardless of the accuracy. Even when it refuses to answer - it explains why it cannot answer. 
For example, it hallucinates numbers when giving answers ("$12 billion") as if this number was derived from somewhere...
This is not surprising because that is the behaviour I encounter since starting using LLMs a couple of years ago - they always answer regardless of the data they have, mostly with a concrete answer, even when completely wrong. This holds also for the best models today.

3. Are there patterns by question_type? Do some types fail more than others?
- The `domain-relevant` Q.s are more yes/no Q.s.<br>
However, with the naive answers we received - I see no significant separation for the model's answers by the questions types...<br>
The one time it was completely wrong was when we asked it for a specific number ("How much ... **in USD million?**") - it just output some random number and was therefore wrong. For other questions, it was more like 50-50% ("Does Corning have positive working capital?") - and its best chance to succeed is to just say "yes" or "no". But it's worth as guessing.

---
## Task 2 - RAG Reminder

### RAG Pipeline Components

#### Indexing (Documents → Chunk + Embed → Vector Store)
**Contribution:** Transforms the raw corpus into a searchable knowledge base by splitting documents into manageable chunks and mapping each chunk to a dense vector in embedding space, so that semantic similarity search becomes possible later. This builds the vector store $D$ that "retrieval" will query against.

**Failure modes:** Poor chunking (e.g., splitting mid-sentence or using chunks too large/small) destroys semantic coherence - a chunk that spans two unrelated topics produces a messy embedding. The embedding model may be domain-mismatched (e.g., a general-purpose encoder on legal or medical text), causing near-duplicate documents to land far apart. Other issues: stale/missing documents, lost metadata (page numbers, titles), OCR errors, or inconsistent preprocessing between indexing time and query time.

**When:** Happens **once, offline** (with periodic re-indexing when the corpus changes). This is the most expensive step per document but amortized across all future queries.

---

#### Retrieval ($\Gamma$: User Query → Top-k Chunks)
**Contribution:** Given a query $q$, embeds it with the *same* encoder used at indexing and searches the vector store $D$ to return the top-$k$ most relevant chunks. This grounds the generator in specific, query-relevant evidence rather than relying purely on parametric memory.

**Failure modes:** Query-document vocabulary mismatch (user asks "how do I cancel?" but docs say "terminate subscription") - pure dense retrieval can miss this, which is why hybrid BM25+dense often helps. Wrong $k$: too small misses key context, too large dilutes the prompt with noise and wastes tokens. Other issues: multi-hop questions that need information spread across chunks, ambiguous queries, or an embedding mismatch between query-time and index-time encoders.

**When:** **Per query** - runs on every user request. Latency-sensitive, so ANN indices (FAISS, HNSW) are typically used instead of exact search.

---

#### Generation ($\Theta$: Query + Retrieved Chunks → Answer)
**Contribution:** An LLM consumes the query plus retrieved context and synthesizes a grounded natural-language answer, ideally citing or quoting the retrieved evidence. This is where retrieved facts become a user-facing response.

**Failure modes:** **Hallucination** even with correct context (the model ignores retrieved chunks and invents facts), or the opposite - the retrieved context *is* wrong/irrelevant and the model faithfully parrots it ("garbage in, garbage out"). Prompt-budget issues: context gets truncated and the crucial chunk is dropped. Also: "lost in the middle" (LLMs under-attend to mid-context chunks, as Yuval stated in class), stale context not reflecting the latest query intent, or tone/format drift from the system prompt.

**When:** **Per query** - one (or more) LLM calls per user request. Usually the dominant cost/latency component of the pipeline.

---
## Task 3 - Embed Documents

In [ ]:
# Task 3 - same filtered corpus as Task 1: first 5 domain-relevant + first 5 novel-generated
filtered_df = df[df["financebench_id"].isin(answers_df["financebench_id"])]
doc_rows = filtered_df.drop_duplicates(subset=["doc_name"], keep="first")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.isdir(VECTORSTORE_DIR):
    vectorstore = FAISS.load_local(
        VECTORSTORE_DIR,
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(
        f"Loaded FAISS index from {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )
else:
    # Load only PDFs for doc_name values in the filtered set; metadata on each page before split
    all_pages = []

    for _, row in doc_rows.iterrows():
        loader = PyPDFLoader(row["doc_link"])
        pages = loader.load()

        for page_number, doc in enumerate(pages):
            doc.metadata = doc.metadata or {}
            doc.metadata["doc_name"] = row["doc_name"]
            doc.metadata["company"] = row["company"]
            doc.metadata["doc_period"] = row["doc_period"]
            doc.metadata["page_number"] = page_number
            all_pages.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
    )
    chunks = text_splitter.split_documents(all_pages)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(VECTORSTORE_DIR)
    print(
        f"Built and saved FAISS index to {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Built and saved FAISS index to 'financebench_rag_faiss' (7559 vectors)


In [ ]:
def check_retrieval(
    row: pd.Series, vectorstore, top_k: int = 5, verbose: bool = True
) -> dict:
    """Retrieve top-k chunks for a dataset row and check doc, page, and evidence overlap."""
    query = row["question"]
    expected_doc = row["doc_name"]

    evidence_texts, evidence_pages = [], set()
    for item in row.get("evidence", []):
        if isinstance(item, dict):
            txt = item.get("evidence_text", "")
            if txt:
                evidence_texts.append(txt)
            pg = item.get("evidence_page_num")
            if pg is not None:
                evidence_pages.add(int(pg))

    retrieved = vectorstore.similarity_search(query, k=top_k)

    doc_hits = [d for d in retrieved if d.metadata.get("doc_name") == expected_doc]
    page_hits = [
        d for d in retrieved if d.metadata.get("page_number") in evidence_pages
    ]
    overlap_hits = [
        d
        for d in retrieved
        if any(
            ev.lower()[:180] in d.page_content.lower()
            or d.page_content.lower()[:180] in ev.lower()
            for ev in evidence_texts
            if ev
        )
    ]

    result = {
        "question": query,
        "expected_doc": expected_doc,
        "evidence_pages": sorted(evidence_pages),
        "doc_match": len(doc_hits),
        "page_match": len(page_hits),
        "evidence_overlap": len(overlap_hits),
        "top_k": top_k,
        "retrieved": retrieved,
    }

    if verbose:
        print("=" * 100)
        print(f"Q: {query}")
        print(f"Expected doc_name: {expected_doc}")
        print(f"Expected evidence pages: {result['evidence_pages'] or 'N/A'}")
        print(
            f"Doc match:          {'YES' if doc_hits else 'NO'} ({len(doc_hits)}/{top_k})"
        )
        print(
            f"Evidence overlap:   {'YES' if overlap_hits else 'NO'} ({len(overlap_hits)}/{top_k})"
        )
        print(
            f"Page match:         {'YES' if page_hits else 'NO'} ({len(page_hits)}/{top_k})"
        )
        print("Top-k retrieved chunks:")
        for rank, d in enumerate(retrieved, start=1):
            print(
                f"  {rank}. {d.metadata.get('doc_name')} | page={d.metadata.get('page_number')}"
            )
            print(f"     {d.page_content[:170].replace(chr(10), ' ')}...")
        print()

    return result


# Task 3 retrieval check on 3 sample questions
retrieval_check = []

for _, row in filtered_df.head(3).iterrows():
    retrieval_check_row = check_retrieval(row, vectorstore)
    retrieval_check.append(retrieval_check_row)

retrieval_check = pd.DataFrame(retrieval_check)

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
Expected doc_name: CORNING_2022_10K
Expected evidence pages: [59]
Doc match:          YES (5/5)
Evidence overlap:   NO (0/5)
Page match:         NO (0/5)
Top-k retrieved chunks:
  1. CORNING_2022_10K | page=101
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  2. CORNING_2022_10K | page=102
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  3. CORNING_2022_10K | page=90
     Corning uses regression analysis or the critical term match method to assess initial hedge effectiveness. Following the inception of a hedging relationship, hedgeeffectiv...
  4. CO

#### Task 3 — Retrieval Observations

**Right document?** Yes — all 3 queries returned **5/5 chunks from the correct company's filing**. This makes sense because the question itself mentions the company name, and that name also appears in the chunks, so the embeddings can easily match them.

**Right evidence text?** No — **0/5** for all 3 questions. I opened the actual PDFs on the evidence pages and found they are mostly balance-sheet tables full of numbers. When we split these pages into 1000-char chunks, the tables get broken apart and lose their structure. The retriever pulls chunks that talk *about* the right topics, but not the exact table rows the dataset annotators marked as evidence.

**Right page?** Mostly no — only **1 out of 15** retrieved chunks came from an expected evidence page (American Water Works, page 81). The balance-sheet pages (59, 60, 80–81) are heavy on numbers and light on regular sentences, so the embedding model doesn't "understand" them well. Instead it prefers pages with more natural language, like company overviews or notes to financial statements, that *mention* working capital in words.

**Takeaway:** The retriever finds the right document easily, but struggles to find the right *page* — especially when the answer lives in a numeric table rather than a text paragraph. This is a known limitation of dense (embedding-based) retrieval. Possible improvements: combining keyword search (BM25) with embeddings, using table-aware chunking, or first filtering by document and then searching within it.

---
## Task 4 - Building a RAG Pipeline